# Snowflake and Feast Feature Store Demo
### Feast documentation
https://docs.feast.dev/ 

### Core concepts

The top-level namespace within Feast is a project. 
- Users define one or more feature views within a project. 
- Each feature view contains one or more features. These features typically relate to one or more entities. 
- A feature view must always have a data source, which in turn is used during the generation of training datasets and when materializing feature values into the online store.

<img src="images\concepts.png" width=500/>


#### Data Source
The data source refers to raw underlying data (e.g. a table in BigQuery). Feast uses a time-series data model to represent data. This data model is used to interpret feature data in data sources in order to build training datasets or when materializing features into an online store.
Dataset
Feast datasets allow for conveniently saving dataframes that include both features and entities to be subsequently used for data analysis and model training. Datasets are created from results of historical retrieval.

Dataset vs Feature View: Feature views contain the schema of data and a reference to where data can be found (through its data source). Datasets are the actual data manifestation of querying those data sources.

Dataset vs Data Source: Datasets are the output of historical retrieval, whereas data sources are the inputs. One or more data sources can be used in the creation of a dataset.

#### Entity
An entity is a collection of semantically related features. Users define entities to map to the domain of their use case.  E.g. account number, customer id, driver id etc.

Entities are typically defined as part of feature views. Entity name is used to reference the entity from a feature view definition and join key is used to identify the physical primary key on which feature values should be stored and retrieved.

#### Feature
A feature is an individual measurable property. It is typically a property observed on a specific entity, but does not have to be associated with an entity. For example, a feature of a customer entity could be the number of transactions they have made on an average month, while a feature that is not observed on a specific entity could be the total number of posts made by all users in the last month.
Features are defined as part of feature views. Since Feast does not transform data, a feature is essentially a schema that only contains a name and a type.

#### Feature View
A feature view is an object that represents a logical group of time-series feature data as it is found in a data source. Feature views consist of zero or more entities, one or more features, and a data source. Feature views allow Feast to model your existing feature data in a consistent way in both an offline (training) and online (serving) environment. 

#### Stream feature view
A stream feature view is an extension of a normal feature view. The primary difference is that stream feature views have both stream and batch data sources, whereas a normal feature view only has a batch data source.

#### Feature Services
A feature service is an object that represents a logical group of features from one or more feature views. Feature Services allows features from within a feature view to be used as needed by an ML model. Users can expect to create one feature service per model version, allowing for tracking of the features used by models.

#### Dataset
Feast datasets allow for conveniently saving dataframes that include both features and entities to be subsequently used for data analysis and model training. Datasets are created from results of historical retrieval.

#### Registry
The Feast registry is where all applied Feast objects (e.g. Feature views, entities, etc) are stored. The registry exposes methods to apply, list, retrieve and delete these objects. The registry is abstraction, with multiple possible implementations.

The registry can be either filed-based or SQL based.

### Feast architecture
<img src="images/feast_architecture2.jpeg" width=800/>
<br><br>
<img src="images/feast_architecture.png" width=500/>

## Installing Feast and initialising a feature store

        pip install 'feast[snowflake]'
        feast init <project-name> -t snowflake

## Define entities, data sources, feature views

**Entity**: Primary key such as an account number, customer id etc. <br>
**Data source**: Underlying data source from where features are retrieved. <br>
**Feature view**: Grouping of features aligned to an online or offline store.


In [1]:
from datetime import datetime, timedelta
import yaml
from feast import FeatureStore, Entity, FeatureService, FeatureView, Field, SnowflakeSource
from feast.types import Float32, Int64
from feast.infra.offline_stores.snowflake_source import SavedDatasetSnowflakeStorage
import pandas as pd

Create feature view for driver orders

In [30]:
# Define an entity for the driver. Entities can be thought of as primary keys used to
# retrieve features. 
driver = Entity(
    name="driver",
    join_keys=["driver_id"],
)

driver_orders_source = SnowflakeSource(
    database=yaml.safe_load(open("feature_store.yaml"))["offline_store"]["database"],
    table="driver_orders",
    warehouse="COMPUTE_WH",
    timestamp_field="event_timestamp"
)

driver_orders_fv = FeatureView(
    name="driver_orders",
    entities=[driver],
    ttl=timedelta(weeks=52),
    schema=[
        Field(name="trip_completed", dtype=Int64),
    ],
    batch_source=driver_orders_source,
    online=True
)


/Users/geoffrey.nightingale@contino.io/Library/Python/3.8/lib/python/site-packages/feast/feature_view.py:285: DeprecationWarning: batch_source and stream_source have been deprecated in favor of `source`.The deprecated fields will be removed in Feast 0.24.
  warnings.warn(


Create feature view for driver stats

In [31]:
project_name = yaml.safe_load(open("feature_store.yaml"))["project"]

# Define an entity for the driver. Entities can be thought of as primary keys used to
# retrieve features. Entities are also used to join multiple tables/views during the
# construction of feature vectors
driver = Entity(
    # Name of the entity. Must be unique within a project
    name="driver",
    # The join keys of an entity describe the storage level field/column on which
    # features can be looked up. The join keys are also used to join feature
    # tables/views when building feature vectors
    join_keys=["driver_id"],
)

# Indicates a data source from which feature values can be retrieved. Sources are queried when building training
# datasets or materializing features into an online store.
driver_stats_source = SnowflakeSource(
    # The Snowflake table where features can be found
    database=yaml.safe_load(open("feature_store.yaml"))["offline_store"]["database"],
    table=f"{project_name}_feast_driver_hourly_stats",
    warehouse="COMPUTE_WH",
    # The event timestamp is used for point-in-time joins and for ensuring only
    # features within the TTL are returned
    timestamp_field="event_timestamp",
    # The (optional) created timestamp is used to ensure there are no duplicate
    # feature rows in the offline store or when building training datasets
    created_timestamp_column="created",
)

# Feature views are a grouping based on how features are stored in either the
# online or offline store.
driver_stats_fv = FeatureView(
    # The unique name of this feature view. Two feature views in a single
    # project cannot have the same name
    name="driver_hourly_stats",
    # The list of entities specifies the keys required for joining or looking
    # up features from this feature view. The reference provided in this field
    # correspond to the name of a defined entity (or entities)
    entities=[driver],
    # The timedelta is the maximum age that each feature value may have
    # relative to its lookup time. For historical features (used in training),
    # TTL is relative to each timestamp provided in the entity dataframe.
    # TTL also allows for eviction of keys from online stores and limits the
    # amount of historical scanning required for historical feature values
    # during retrieval
    ttl=timedelta(weeks=52),
    # The list of features defined below act as a schema to both define features
    # for both materialization of features into a store, and are used as references
    # during retrieval for building a training dataset or serving features
    schema=[
        Field(name="conv_rate", dtype=Float32),
        Field(name="acc_rate", dtype=Float32),
        # Field(name="avg_daily_trips", dtype=Int64),
    ],
    # Batch sources are used to find feature values. In the case of this feature
    # view we will query a source table on Redshift for driver statistics
    # features
    batch_source=driver_stats_source,
    online=True
)


Create another feature view for average daily trips 

In [32]:
driver_stats_source = SnowflakeSource(
    database=yaml.safe_load(open("feature_store.yaml"))["offline_store"]["database"],
    table=f"{project_name}_feast_driver_hourly_stats",
    warehouse="COMPUTE_WH",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

driver_trips_fv = FeatureView(
    name="driver_avg_daily_trips",
    entities=[driver],
    ttl=timedelta(weeks=52),
    schema=[
        Field(name="avg_daily_trips", dtype=Int64),
    ],
    batch_source=driver_stats_source,
    online=True
)


In [33]:
# Create a feature service
driver_stats_fs = FeatureService(name="driver_activity", features=[driver_stats_fv, driver_trips_fv, driver_orders_fv])

## Apply changes made to the feature store
Apply changes made to the feature store.

In [35]:
# Load the feature store from the current path
fs = FeatureStore(repo_path=".")

# Deploy the feature store to Snowflake
print("Deploying feature store to Snowflake...")
fs.apply([driver, driver_stats_fv, driver_trips_fv, driver_orders_fv, driver_stats_fs])

Deploying feature store to Snowflake...


## Retrieving data from the feature store

We can retrieve features across numerous feature views:

In [11]:
# Define feature views and features to retrieve
features = ["driver_hourly_stats:conv_rate", 
            "driver_hourly_stats:acc_rate", 
            "driver_avg_daily_trips:avg_daily_trips",
            "driver_orders:trip_completed"]

In [19]:
# Create an entity dataframe. This is the dataframe that will be enriched with historical features
entity_df = pd.DataFrame(
    {
        "event_timestamp": [
            pd.Timestamp(dt, unit="ms", tz="UTC").round("ms")
            for dt in pd.date_range(
                start=datetime.now() - timedelta(days=3),
                end=datetime.now(),
                periods=3,
            )
        ],
        "driver_id": [1001, 1002, 1003],
    }
)

print(entity_df.head())

                   event_timestamp  driver_id
0 2022-07-19 13:07:01.519000+00:00       1001
1 2022-07-21 01:07:01.519000+00:00       1002
2 2022-07-22 13:07:01.519000+00:00       1003


In [21]:
# Retrieve historical features by joining the entity dataframe to the Snowflake table source
training_df = fs.get_historical_features(
    features=features, entity_df=entity_df
).to_df()

print(training_df.head())


          event_timestamp  driver_id  conv_rate  acc_rate  avg_daily_trips  \
0 2022-07-22 13:07:01.519       1003   0.507542  0.552442              153   
1 2022-07-21 01:07:01.519       1002   0.025624  0.374785              552   
2 2022-07-19 13:07:01.519       1001   0.913352  0.466293              975   

   trip_completed  
0             NaN  
1             NaN  
2             NaN  


## Retrieve data from a feature service

Alternatively we can retrieve features from a feature service:

In [16]:
feature_service = fs.get_feature_service("driver_activity")
training_df = fs.get_historical_features(features=feature_service, entity_df=entity_df).to_df()

print(training_df.head())

          event_timestamp  driver_id  conv_rate  acc_rate  avg_daily_trips  \
0 2022-07-22 12:58:16.439       1003   0.507542  0.552442              153   
1 2022-07-21 00:58:16.439       1002   0.025624  0.374785              552   
2 2022-07-19 12:58:16.439       1001   0.034525  0.450478               51   

   trip_completed  
0             NaN  
1             NaN  
2             NaN  


## Save data into Snowflake
For example, say we want to save a copy of a training dataset that we have used to build a model.

In [17]:
df = fs.get_historical_features(features=feature_service, entity_df=entity_df)

dataset = fs.create_saved_dataset(
    from_=training_df,
    name='driver_training_dataset',
    storage=SavedDatasetSnowflakeStorage(table_ref='my_training_dataset'),
    tags={'author': 'geoff'}
)

/Users/geoffrey.nightingale@contino.io/Library/Python/3.8/lib/python/site-packages/feast/feature_store.py:1115: RuntimeWarning: Saving dataset is an experimental feature. This API is unstable and it could and most probably will be changed in the future. We do not guarantee that future changes will maintain backward compatibility.
  warnings.warn(


## Load data into the online feature store
Materlialising the latest data for entities int the online feature store.

In [27]:
fs.materialize_incremental(end_date=datetime.now())

Materializing 3 feature views to 2022-07-22 14:53:31+01:00 into the sqlite online store.

driver_hourly_stats from 2022-07-22 14:00:05+01:00 to 2022-07-22 14:53:31+01:00:


0it [00:00, ?it/s]


driver_avg_daily_trips from 2022-07-22 14:00:05+01:00 to 2022-07-22 15:53:31+01:00:


0it [00:00, ?it/s]


driver_orders from 2022-07-22 14:00:05+01:00 to 2022-07-22 15:53:31+01:00:


0it [00:00, ?it/s]


## Get data from online feature store

In [26]:
online_features = fs.get_online_features(
    features=features, entity_rows=[{"driver_id": 1001}, {"driver_id": 1002}],
).to_df()

online_features

,driver_id,conv_rate,acc_rate,avg_daily_trips,trip_completed
0,1001,0.913352,0.466293,975,None
1,1002,0.025624,0.374785,552,None


## Feast UI

In [28]:
!feast ui -h localhost

/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/site-packages/feast/feature_store.py:2229: RuntimeWarning: The Feast UI is an experimental feature. We do not guarantee that future changes will maintain backward compatibility.
  warnings.warn(
INFO:     Started server process [33212]
07/22/2022 02:54:29 PM uvicorn.error INFO: Started server process [33212]
INFO:     Waiting for application startup.
07/22/2022 02:54:29 PM uvicorn.error INFO: Waiting for application startup.
INFO:     Application startup complete.
07/22/2022 02:54:29 PM uvicorn.error INFO: Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('::1', 8888, 0, 0): address already in use
07/22/2022 02:54:29 PM uvicorn.error ERROR: [Errno 48] error while attempting to bind on address ('::1', 8888, 0, 0): address already in use
INFO:     Waiting for application shutdown.
07/22/2022 02:54:29 PM uvicorn.error INFO: Waiting for application shutdown.
INFO:     Application 

## Feast CLI

In [29]:
!feast ui -h 0.0.0.0 -p 9090 -r 5

/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/site-packages/feast/feature_store.py:2229: RuntimeWarning: The Feast UI is an experimental feature. We do not guarantee that future changes will maintain backward compatibility.
  warnings.warn(
INFO:     Started server process [33222]
07/22/2022 02:54:35 PM uvicorn.error INFO: Started server process [33222]
INFO:     Waiting for application startup.
07/22/2022 02:54:35 PM uvicorn.error INFO: Waiting for application startup.
INFO:     Application startup complete.
07/22/2022 02:54:35 PM uvicorn.error INFO: Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9090 (Press CTRL+C to quit)
07/22/2022 02:54:35 PM uvicorn.error INFO: Uvicorn running on http://0.0.0.0:9090 (Press CTRL+C to quit)
INFO:     127.0.0.1:62975 - "GET /projects-list.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:62975 - "GET /registry HTTP/1.1" 200 OK
INFO:     127.0.0.1:62975 - "GET /registry HTTP/1.1" 200 OK
INFO:     127.0.0.1:6297

In [112]:
!feast feature-views list

NAME                    ENTITIES    TYPE
driver_hourly_stats     {'driver'}  FeatureView
driver_avg_daily_trips  {'driver'}  FeatureView
driver_orders           {'driver'}  FeatureView
^C
Exception ignored in: <module 'threading' from '/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/threading.py'>
Traceback (most recent call last):
  File "/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/threading.py", line 1440, in _shutdown
    atexit_call()
  File "/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/concurrent/futures/thread.py", line 31, in _python_exit
    t.join()
  File "/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/threading.py", line 1053, in join
    self._wait_for_tstate_lock()
  File "/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/threading.py", line 1073, in _wait_for_tstate_lock
    if lock.acquire(block, timeout):
KeyboardInterrupt: 


In [113]:
!feast entities list

NAME    DESCRIPTION    TYPE
driver                 ValueType.UNKNOWN
^C
Exception ignored in: <module 'threading' from '/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/threading.py'>
Traceback (most recent call last):
  File "/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/threading.py", line 1440, in _shutdown
    atexit_call()
  File "/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/concurrent/futures/thread.py", line 31, in _python_exit
    t.join()
  File "/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/threading.py", line 1053, in join
    self._wait_for_tstate_lock()
  File "/Users/geoffrey.nightingale@contino.io/opt/anaconda3/lib/python3.9/threading.py", line 1073, in _wait_for_tstate_lock
    if lock.acquire(block, timeout):
KeyboardInterrupt: 


In [32]:
!feast registry-dump

{
  "dataSources": [
    {
      "name": "DRIVER_ORDERS",
      "snowflakeOptions": {
        "database": "FEAST_TEST",
        "schema": "PUBLIC",
        "table": "DRIVER_ORDERS",
        "warehouse": "COMPUTE_WH"
      },
      "timestampField": "EVENT_TIMESTAMP",
      "type": "BATCH_SNOWFLAKE"
    },
    {
      "name": "driver_orders",
      "snowflakeOptions": {
        "database": "FEAST_TEST",
        "schema": "PUBLIC",
        "table": "driver_orders",
        "warehouse": "COMPUTE_WH"
      },
      "timestampField": "event_timestamp",
      "type": "BATCH_SNOWFLAKE"
    },
    {
      "createdTimestampColumn": "created",
      "name": "feast_test_feast_driver_hourly_stats",
      "snowflakeOptions": {
        "database": "FEAST_TEST",
        "schema": "PUBLIC",
        "table": "feast_test_feast_driver_hourly_stats",
        "warehouse": "COMPUTE_WH"
      },
      "timestampField": "event_timestamp",
      "type": "BATCH_SNOWFLAKE"
    }
  ],
  "entities": [
    {
      